# Segmentation – Day 12: Methodological Design

**Student Name:** Traver Dinten
**Country:** Switzerland
**Semester term:** FS26

## Theoretical Foundation and Method Choice
*Focus: principled justification aligned with the use case*

This investigation applies **Otsu's global thresholding** to separate roasted coffee beans from a white background within the context of Swiss coffee quality inspection. Otsu's method automatically selects a threshold that minimizes the intra-class variance between foreground and background pixel populations, assuming a bimodal intensity distribution. This assumption is well-satisfied for the coffee bean image: the histogram (Day 11) exhibits two clearly separated populations — dark beans (intensity 0.05–0.75) and bright background (intensity > 0.85) — with a distinct valley around 0.80. Because the illumination in the selected image is approximately uniform (controlled studio lighting, flat white surface), a single global threshold is expected to provide consistent separation across the entire image without the added complexity of adaptive methods.

## Morphological Refinement

After thresholding, the binary segmentation mask will be refined using morphological operations to improve object boundary quality and handle segmentation artifacts:

**1. Morphological Opening (Erosion → Dilation):**
- **Principle:** Opening removes small foreground noise (isolated pixels or thin connections) by first eroding and then dilating. It eliminates structures smaller than the structuring element while preserving the overall shape and size of larger objects.
- **Intended effect:** Remove small spurious foreground regions caused by shadow artifacts or noise at bean edges, and potentially break thin connections between touching beans. The operation preserves the elliptical bean shapes because their area is much larger than the structuring element.

**2. Morphological Closing (Dilation → Erosion):**
- **Principle:** Closing fills small holes and gaps within foreground objects by first dilating and then eroding. It reconnects regions that were separated by narrow background intrusions.
- **Intended effect:** Fill small holes within individual beans caused by the central crease (a dark line along the bean's long axis that may be thresholded as background). Without closing, the crease could split a single bean into two foreground regions or create internal holes affecting area measurements.

## Key Parameters and Expected Effects

Three key parameters are identified that control the segmentation outcome:

**1. Threshold Value (Otsu-derived, with manual variations):**
- **Function:** Determines the intensity boundary separating foreground (beans) from background. Otsu's method computes this automatically, but manual offsets will be tested to assess sensitivity.
- **Expected effect:** The Otsu threshold is expected to fall near the histogram valley (~0.80). A lower threshold will classify more intermediate-intensity pixels as foreground, potentially including shadow regions and expanding bean boundaries. A higher threshold will exclude marginal pixels, potentially under-segmenting lighter bean surfaces or edges.

**2. Morphological Kernel Size (disk radius for opening/closing):**
- **Function:** Controls the scale of the structuring element used in morphological operations. Larger kernels affect larger spatial features.
- **Expected effect:** A small kernel (radius 2–3 px) will remove only fine-scale noise while preserving bean boundaries. A larger kernel (radius 5–7 px) will more aggressively smooth boundaries and may break connections between touching beans, but risks eroding small beans or altering their measured area and perimeter.

**3. Minimum Object Area (for connected component filtering):**
- **Function:** After labeling connected components, objects smaller than a minimum area threshold are discarded as noise.
- **Expected effect:** This parameter filters out small spurious detections (shadow fragments, background noise) without affecting actual beans. An appropriate minimum area should be set well below the smallest expected bean size to avoid discarding real objects.

## Skeletonization

For one selected bean (the largest segmented object), the binary shape will be reduced to its **morphological skeleton** — a one-pixel-wide representation that preserves the topological structure and elongation of the object.

**Why this is meaningful:** Coffee beans are elongated elliptical objects with a characteristic central crease. The skeleton of a well-segmented bean should approximate a single line segment along the bean's long axis, reflecting its elongated shape. The number of skeleton pixels provides a scale-independent measure of the object's length. In quality control, deviations from this expected linear skeleton structure (e.g., branching, fragmentation) would indicate segmentation artifacts or atypical bean shapes. Comparing the skeleton pixel count across configurations also serves as a secondary indicator of how morphological operations affect the internal structure representation.

## Experimental Design and Configurations

To evaluate the impact of threshold selection and morphological refinement on segmentation quality, a controlled experimental design is defined with a baseline and two variations. This avoids loosely connected parameter sweeps and enables targeted comparison.

**Baseline Configuration:**
- Threshold: Otsu's automatic threshold
- Morphological operations: opening (disk, radius 3) → closing (disk, radius 3)
- Minimum object area: 100 px
- *Justification:* Otsu provides an objective, data-driven threshold. A disk radius of 3 is small enough to preserve bean shape while removing fine noise and filling the central crease. The 100 px minimum area filter removes small artifacts without risking exclusion of real beans.

**Variation 1: Threshold Sensitivity**
The threshold is shifted by ±0.05 from the Otsu value while keeping morphological parameters fixed at baseline.
- **Configurations:** $T_{Otsu} - 0.05$, $T_{Otsu}$ (baseline), $T_{Otsu} + 0.05$
- *Justification:* This tests sensitivity to threshold placement within the histogram valley. A ±0.05 shift is expected to produce measurable differences in boundary inclusion (shadow pixels at lower threshold) or exclusion (light bean surfaces at higher threshold) without collapsing the segmentation entirely.

**Variation 2: Morphological Kernel Size**
The disk radius is varied while keeping the threshold fixed at Otsu.
- **Configurations:** radius $\in \{1, 3, 5\}$
- *Justification:* Radius 1 applies minimal morphological refinement (nearly raw thresholding), radius 3 is the baseline, and radius 5 applies stronger smoothing. This isolates the effect of morphological scale on bean separation, boundary smoothness, and measured object properties.

This structured set of 5 unique configurations (3 threshold + 3 kernel, with the baseline shared) enables systematic assessment of the two primary factors affecting segmentation quality.

## Object Extraction and Property Measurement

**Object extraction:** After applying thresholding and morphological refinement, individual objects are extracted using **connected component labeling** (`skimage.measure.label`). Each connected foreground region is assigned a unique label. Objects with area below the minimum threshold are removed.

**Object properties (2–3 measures):**

The following properties will be measured for each segmented object using `skimage.measure.regionprops`:

1. **Area** (pixels): The total number of foreground pixels in the object. Area is the most direct measure of bean size and is essential for detecting size consistency within a batch — a key grading criterion in coffee quality control.

2. **Perimeter** (pixels): The length of the object boundary. Perimeter is sensitive to boundary roughness introduced by thresholding artifacts or morphological operations. The ratio of area to perimeter also provides an indirect shape regularity indicator.

3. **Eccentricity** (dimensionless, 0–1): The eccentricity of the ellipse that has the same second moments as the object. A value of 0 indicates a circle, while values approaching 1 indicate increasingly elongated shapes. Coffee beans are expected to have moderate eccentricity (approximately 0.6–0.8) due to their elliptical shape. This measure helps identify merged objects (which would appear more circular) or broken fragments (which may have atypical eccentricity).

## Methodological Limitations and Risk Factors

This approach assumes that the image has approximately uniform illumination and that the intensity histogram is clearly bimodal — both conditions satisfied in the selected studio photograph but potentially violated in real production environments with uneven lighting. The method is expected to be reliable when beans are spatially separated with sufficient background gaps, but may become unreliable when multiple beans are in direct contact, as connected component labeling will merge them into a single object. In this coffee quality inspection use case, the primary risk factors are: (1) the bean crease creating internal holes if closing is insufficient, (2) shadows expanding apparent bean area if the threshold is too low, and (3) touching beans being counted as a single object, which directly affects the counting accuracy that is central to the defined objective.